# CRIVO — Exportar o 46M (6b v4) para o chat (NumPy int8)

Converte a versão da fase 6b v4 que vai para o chat (46M, 24 camadas, contexto 1024, SHA `45825352…`) para o formato do site: NumPy com int8 por linha (`scripts/quantizar_q8.py`, o mesmo da v4 F). Confere NumPy int8 × PyTorch e divide o arquivo em partes de até 3,5 MB no Drive, para o assistente pegar.

Vai para o chat **em teste, por decisão do criador** ("Pode colocar no chat", 14/10/2026). Na régua de sentido ele **empata** com a v4 F, não a supera (`experimentos/fase6b_conversa_humana/RESULTADO.md`).

**Não precisa de GPU.** Ambiente de execução → Executar tudo (uns 5 minutos). Resultado em `CRIVO/conversa46m-v4-numpy/`.

In [ ]:
# Célula 0 — Drive e os pesos do 46M (6b v4)
import os, sys, json, shutil, subprocess, hashlib, glob
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
MEU = Path("/content/drive/MyDrive")
SHA = "458253529e07604a5438ee772fde110e3dc9569604d740ee91990e219408651a"
def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
ORIGEM = CRIVO = None
for crivo in candidatas:
    for p in sorted((crivo / "transformer-crivo-46m-conversa-v4").glob("conversa_*/pesos.pt")):
        if sha(p) == SHA:
            ORIGEM, CRIVO = p, crivo
            break
    if ORIGEM:
        break
if ORIGEM is None:
    raise SystemExit("ERRO: não achei o pesos.pt da 6b v4 (sha 45825352…) em CRIVO/transformer-crivo-46m-conversa-v4/. "
                     "Rode numa conta que veja a pasta CRIVO (hr87676620 ou henriquenbk18).")
LOCAL = Path("/content/c46m")
shutil.rmtree(LOCAL, ignore_errors=True)
LOCAL.mkdir()
for n in ("pesos.pt", "tokenizer.json", "formato_fase6.py"):
    shutil.copyfile(ORIGEM.parent / n, LOCAL / n)
print("pesos:", ORIGEM, flush=True)


In [ ]:
# Célula 1 — exportar (float16), int8 por linha e conferir NumPy × PyTorch
REPO = Path("/content/CRIVO")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
for p in (str(REPO), str(REPO / "scripts"), str(REPO / "experimentos" / "fase6b_conversa_humana")):
    sys.path.insert(0, p)
import numpy as np, torch
import exportar_pontuador, quantizar_q8, formato_fase6
import linguagem_profunda as lp
from realizador_17m import RealizadorNumpy
from tokenizers import Tokenizer
exportar_pontuador.exportar(LOCAL, LOCAL)
(LOCAL / "pesos_numpy.npz").rename(LOCAL / "pesos_f16.npz")
quantizar_q8.quantizar(LOCAL / "pesos_f16.npz", LOCAL / "pesos_numpy.npz")
NPZ = LOCAL / "pesos_numpy.npz"
print("int8:", round(NPZ.stat().st_size / 1e6, 1), "MB | sha256", sha(NPZ), flush=True)

modelo, _, _ = lp.carregar(LOCAL, dispositivo="cpu")
modelo.eval()
TOK = Tokenizer.from_file(str(LOCAL / "tokenizer.json"))
ESP = {t: TOK.token_to_id(t) for t in ("<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>")}
cod = lambda t: TOK.encode(t, add_special_tokens=False).ids
def numpy_de(npz):
    m = RealizadorNumpy.__new__(RealizadorNumpy)
    m.np = np
    m.p, meta = quantizar_q8.carregar(npz)
    m.camadas, m.cabecas, m.contexto = meta["camadas"], meta["cabecas"], meta["contexto"]
    return m
QUEM = "O CRIVO é uma inteligência artificial curiosa e calma, que ama aprender, diz a verdade com carinho, cuida das pessoas e enxerga que tudo está conectado."
falas = [("eae tudo bem", "Tudo bem, sim! E você?"), ("kkkkk", "Haha! O que foi?"), ("quem é você?", "Sou o CRIVO."),
         ("tô triste hoje", "Sinto muito. Quer me contar?"), ("qual é o meu nome?", "Seu nome é Ana."),
         ("me fala sobre Júpiter", "Júpiter é o maior planeta do Sistema Solar."), ("você é burro", "Entendo a irritação."),
         ("como faço arroz?", "Não sei te dizer.")]
r = {}
for nome, npz in (("float16", LOCAL / "pesos_f16.npz"), ("int8", NPZ)):
    m = numpy_de(npz)
    iguais = total = 0; dif = 0.0
    for f, a in falas:
        ctx = {"quem": QUEM, "lembrancas": ["a pessoa disse: meu nome é Ana"]}
        ids, _ = formato_fase6.montar(cod, ESP, ctx, [{"papel": "usuario", "texto": f}, {"papel": "assistente", "texto": a}])
        with torch.no_grad():
            ref = modelo(torch.tensor([ids]))[0][0].numpy()
        lg = m._passo(ids, 0, {}) @ m.p["embedding.weight"].T
        iguais += int((ref.argmax(-1) == lg.argmax(-1)).sum()); total += len(ids)
        dif = max(dif, float(abs(ref - lg).max()))
    r[nome] = {"top1_igual": round(iguais / total, 4), "maior_diferenca_logit": round(dif, 3)}
print("NumPy × PyTorch:", json.dumps(r), flush=True)
if r["float16"]["top1_igual"] < 0.99 or r["int8"]["top1_igual"] < 0.95:
    raise SystemExit("ERRO: o NumPy não bate com o PyTorch. Mande o print para o assistente.")


In [ ]:
# Célula 2 — partes de até 3,5 MB no Drive (o conector do Drive não entrega arquivos maiores)
SAIDA = CRIVO / "conversa46m-v4-numpy"
SAIDA.mkdir(parents=True, exist_ok=True)
for antigo in SAIDA.glob("parte_*"):
    antigo.unlink()
dados = NPZ.read_bytes()
TAM = 3584 * 1024
partes = []
for i in range(0, len(dados), TAM):
    nome = "parte_%02d.bin" % (i // TAM)
    (SAIDA / nome).write_bytes(dados[i:i + TAM])
    partes.append({"nome": nome, "sha256": hashlib.sha256(dados[i:i + TAM]).hexdigest()})
manifesto = {"pesos_origem_sha256": SHA, "npz_sha256": sha(NPZ), "npz_bytes": len(dados), "f16_sha256": sha(LOCAL / "pesos_f16.npz"),
             "partes": partes, "tokenizer_sha256": sha(LOCAL / "tokenizer.json"), "formato_fase6_sha256": sha(LOCAL / "formato_fase6.py"),
             "paridade": r}
(SAIDA / "manifesto_46m_v4.json").write_text(json.dumps(manifesto, indent=1))
print(json.dumps({k: v for k, v in manifesto.items() if k != "partes"}, indent=1), "\n", len(partes), "partes")
print("\nPRONTO: avise o assistente, ele pega as partes no Drive.", flush=True)
